Step 1: 환경 설정

# installing pre-requisites
!pip install -r requirements.txt

Step 2: 필요한 종속성 가져오기

In [1]:
import asyncio
import operator
from typing import Literal

import httpx
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from strands import Agent, tool

@tool 데코레이터로 간단한 calculator 도구를 정의

In [2]:
_OPS = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
    "**": operator.pow,
}

@tool
def calculator(a: float, b: float, op: Literal["+", "-", "*", "/", "**"]) -> float:
    """Apply an arithmetic operator to two numbers.

    Args:
        a: Left operand.
        b: Right operand.
        op: One of "+", "-", "*", "/", "**".
    """
    return _OPS[op](a, b)

In [3]:
nest_asyncio.apply()

Step 1 - stream_async로 에이전트 생성 및 호출, callback handler 없이 에이전트 초기화

In [4]:
agent = Agent(
    model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    tools=[calculator], 
    callback_handler=None)

스트리밍된 에이전트 이벤트를 반복하는 비동기 함수

In [5]:
async def process_streaming_response():
    agent_stream = agent.stream_async("Calculate 2+2")
    async for event in agent_stream:
        print(event)

에이전트 실행

In [6]:
asyncio.run(process_streaming_response())

{'init_event_loop': True}
{'start': True}
{'start_event_loop': True}
{'event': {'messageStart': {'role': 'assistant'}}}
{'event': {'contentBlockStart': {'start': {'toolUse': {'toolUseId': 'tooluse_fDd5LyvLAyUHNzhJG0R0DB', 'name': 'calculator', 'type': 'tool_use'}}, 'contentBlockIndex': 0}}}
{'event': {'contentBlockDelta': {'delta': {'toolUse': {'input': ''}}, 'contentBlockIndex': 0}}}
{'delta': {'toolUse': {'input': ''}}, 'current_tool_use': {'toolUseId': 'tooluse_fDd5LyvLAyUHNzhJG0R0DB', 'name': 'calculator', 'input': ''}, 'agent': <strands.agent.agent.Agent object at 0x7fd8bc1e8230>, 'event_loop_cycle_id': UUID('ccdff462-3cbe-4d7f-ba36-5141896a897c'), 'request_state': {}, 'event_loop_cycle_trace': <strands.telemetry.metrics.Trace object at 0x7fd8af5b01a0>, 'event_loop_cycle_span': NonRecordingSpan(SpanContext(trace_id=0x00000000000000000000000000000000, span_id=0x0000000000000000, trace_flags=0x00, trace_state=[], is_remote=False))}
{'event': {'contentBlockDelta': {'delta': {'toolUse

이벤트 루프 수명 주기 추적

In [7]:
#Step 1 - 스트리밍된 에이전트 이벤트를 반복하는 비동기 함수
async def process_streaming_response():
    agent_stream = agent.stream_async("What is the capital of France and what is 42+7?")
    async for event in agent_stream:
        # 이벤트 루프 수명 주기 추적
        if event.get("init_event_loop", False):
            print("🔄 Event loop initialized")
        elif event.get("start_event_loop", False):
            print("▶️ Event loop cycle starting")
        elif event.get("start", False):
            print("📝 New cycle started")
        elif "message" in event:
            print(f"📬 New message created: {event['message']['role']}")
        elif event.get("force_stop", False):
            print(
                f"🛑 Event loop force-stopped: {event.get('force_stop_reason', 'unknown reason')}"
            )

        # Step 2 - 도구 사용 추적
        if "current_tool_use" in event and event["current_tool_use"].get("name"):
            tool_name = event["current_tool_use"]["name"]
            print(f"🔧 Using tool: {tool_name}")

        # Step 3 - 출력을 깔끔하게 유지하기 위해 텍스트의 일부만 표시
        if "data" in event:
            # 데모 목적으로 각 청크의 처음 20자만 표시
            data_snippet = event["data"][:20] + (
                "..." if len(event["data"]) > 20 else ""
            )
            print(f"📟 Text: {data_snippet}")

    return event["result"]


# Step 4 - 에이전트 실행
await process_streaming_response()

🔄 Event loop initialized
📝 New cycle started
▶️ Event loop cycle starting
📟 Text: The
📟 Text:  capital of France i...
📟 Text: Paris**.
🔧 Using tool: calculator
🔧 Using tool: calculator
🔧 Using tool: calculator
🔧 Using tool: calculator
🔧 Using tool: calculator
🔧 Using tool: calculator
📬 New message created: assistant
📬 New message created: user
📝 New cycle started
📝 New cycle started
▶️ Event loop cycle starting
📟 Text: And
📟 Text:  42 + 7 =
📟 Text:  **49**.
📬 New message created: assistant


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'And 42 + 7 = **49**.'}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_e6aFnRqb0DwTFC5AmXaaqg', 'name': 'calculator', 'input': {'a': 42, 'b': 7, 'op': '+'}}, call_count=2, success_count=2, error_count=0, total_time=0.0007009506225585938)}, cycle_durations=[1.9062912464141846, 1.2446582317352295], traces=[<strands.telemetry.metrics.Trace object at 0x7fd8af5b01a0>, <strands.telemetry.metrics.Trace object at 0x7fd8bc1badb0>, <strands.telemetry.metrics.Trace object at 0x7fd8affebe30>, <strands.telemetry.metrics.Trace object at 0x7fd8afcc7e30>], accumulated_usage={'inputTokens': 3244, 'outputTokens': 211, 'totalTokens': 3455}, accumulated_metrics={'latencyMs': 6958}), state={})

FastAPI 통합

Step 1 - 도구 정의

In [12]:
@tool
def weather_forecast(city: str, days: int = 3) -> str:
    return f"Weather forecast for {city} for the next {days} days..."

# FastAPI 앱
app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/stream")
async def stream_response(request: PromptRequest):
    async def generate():
        agent = Agent(
            model="us.anthropic.claude-sonnet-5",
            tools=[calculator, weather_forecast], callback_handler=None)
        try:
            async for event in agent.stream_async(request.prompt):
                if "data" in event:
                    yield event["data"]
        except Exception as e:
            yield f"Error: {str(e)}"

    return StreamingResponse(generate(), media_type="text/plain")

Step 2 - 서버를 차단하지 않고 시작하는 함수

In [13]:
async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

Step 3 - 백그라운드 작업으로 서버 실행

In [14]:
if "server_task" not in globals():
    server_task = asyncio.create_task(start_server())
    await asyncio.sleep(0.1)  # 서버가 시작할 시간을 줍니다

print("✅ Server is running at http://0.0.0.0:8001")

✅ Server is running at http://0.0.0.0:8001


Step 4 - FastAPI 에이전트 호출

In [16]:
async def fetch_stream():
    async with httpx.AsyncClient() as client:
        async with client.stream(
            "POST",
            "http://0.0.0.0:8001/stream",
            json={"prompt": "서울의 날씨는 어때?"},
        ) as response:
            async for line in response.aiter_lines():
                if line.strip():  # 빈 줄 건너뛰기
                    print("Received:", line)

await fetch_stream()

Received: 서울의 날씨 정보를 조회했지만, 결과에 "향후 3일간의 서울 날씨 예보"라는 안내 문구만 있고 기온이나 강수 같은 실제 내용은 없었습니다. 그래서 날씨를 알려드릴 수 없습니다.
Received: 다시 조회해 볼까요? 예보 기간을 다르게 해서 조회할 수도 있습니다. 급하시면 기상청이나 날씨 앱을 확인하시는 게 더 정확합니다.


Method 2 - 스트리밍을 위한 Callback Handlers

Step 1 - 스트림 데이터 처리

In [17]:
def custom_callback_handler(**kwargs):
    # Process stream data
    if "data" in kwargs:
        print(f"MODEL OUTPUT: {kwargs['data']}")
    elif "current_tool_use" in kwargs and kwargs["current_tool_use"].get("name"):
        print(f"\nUSING TOOL: {kwargs['current_tool_use']['name']}")

Step 2 - 사용자 정의 callback handler로 에이전트 생성

In [18]:
agent = Agent(
    model="us.anthropic.claude-sonnet-5",
    tools=[calculator], callback_handler=custom_callback_handler)

agent("Calculate 2+2")


USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator

USING TOOL: calculator
MODEL OUTPUT: 2
MODEL OUTPUT:  + 2 = 4


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '2 + 2 = 4'}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_dzWCOEdyohw0RX8vMImILQ', 'name': 'calculator', 'input': {'a': 2, 'b': 2, 'op': '+'}}, call_count=1, success_count=1, error_count=0, total_time=0.0003020763397216797)}, cycle_durations=[1.0523302555084229], traces=[<strands.telemetry.metrics.Trace object at 0x7fd8ae342450>, <strands.telemetry.metrics.Trace object at 0x7fd8a7c1b6e0>], accumulated_usage={'inputTokens': 1315, 'outputTokens': 110, 'totalTokens': 1425}, accumulated_metrics={'latencyMs': 2757}), state={})